This is for 2026-02-09 version of KTU LT hate corpus.

Performs cleanup. Converts to chunked corpus. Aggregates into binary labels.

In [ ]:
%load_ext autoreload
%autoreload all

#import typing stuff
from typing import List, Dict, Tuple, Union, Any, Self

#import notebook helpers
from IPython.display import display,  clear_output
from tqdm import tqdm
import builtins

#import libraries
import sys, os.path
importPath = os.path.abspath('../../common')
if not importPath in sys.path:
    sys.path.append(importPath)

#define base paths
origBasePath = os.path.abspath("./original/ktu/LT_HS_2026-02-09")
print(origBasePath)

preprocBasePath = os.path.abspath("./preproc/ktu/LT_HS_2026-02-09-binary-mixed")
print(preprocBasePath)

In [ ]:
#define file paths
inFilePathsLoaded = [
	(os.path.join(origBasePath, "DATASET No. 1 Ethnicity _ nationality _ race_HS.v1.xlsx"), "ethnicity"),
	(os.path.join(origBasePath, "DATASET No. 2 S. Orientation _ gender_HS.v1.xlsx"), "gender"),
	(os.path.join(origBasePath, "DATASET No. 3 Countries_HS.v1.xlsx"), "countries"),
	(os.path.join(origBasePath, "DATASET No. 4 Political_views_HS.v1.xlsx"), "political"),
	(os.path.join(origBasePath, "DATASET No. 5 Religion__HS.v1.xlsx"), "religion")
]
# display(inFilePathsLoaded)

inFilePathsNeutral = [
	(os.path.join(origBasePath, "DATASET No. 1 Ethnicity _ nationality _ race_N.v1.xlsx"), "ethnicity"),
	(os.path.join(origBasePath, "DATASET No. 2 S. Orientation _ gender_N.v1.xlsx"), "gender"),
	(os.path.join(origBasePath, "DATASET No. 3 Countries_N.v1.xlsx"), "countries"),
	(os.path.join(origBasePath, "DATASET No. 4 Political_views_N.v1.xlsx"), "political"),
	(os.path.join(origBasePath, "DATASET No. 5 Religion__N.v1.xlsx"), "religion"),
]
# display(inFilePathsNeutral)

In [ ]:
#read inputs, print headers for quick sanity check

import polars as pl

dfLoadedAll = [
    pl.read_excel(path, sheet_name="in").with_columns(pl.lit(type).alias("type")) 
    for path, type in inFilePathsLoaded
]
display([(os.path.basename(dfPath), df.columns) for (dfPath, type), df in zip(inFilePathsLoaded, dfLoadedAll)])

dfNeutralAll = [
    pl.read_excel(path, sheet_name="in").with_columns(pl.lit(type).alias("type")) 
    for path, type in inFilePathsNeutral
]
display([(os.path.basename(dfPath), df.columns) for (dfPath, type), df in zip(inFilePathsNeutral, dfNeutralAll)])

In [ ]:
#put everything into one table

def levelNameToInt(name : str) -> int:
    names = {
        "level 1" : 1,
        "level 2" : 2,
        "level 3" : 3,
        "level 4" : 4
    }
    name = name.strip().lower()
    if name in names:
        return names[name]
    else:
        raise AssertionError(f"Unsuported level name '{name}'.")

#add loaded texts
colText = []
colLvlEthnicityHate = []
colLvlGenderHate = []
colLvlCountriesHate = []
colLvlPoliticalHate = []
colLvlReligionHate = []

for df in dfLoadedAll:
    for row in df.iter_rows(named=True):
        colText.append(row["Comment"].strip())

        colLvlEthnicityHate.append(0)
        colLvlGenderHate.append(0)
        colLvlCountriesHate.append(0)
        colLvlPoliticalHate.append(0)
        colLvlReligionHate.append(0)

        if row["type"] == "ethnicity":
            lvl = levelNameToInt(row["HS Level"])
            colLvlEthnicityHate[-1] = lvl
        elif row["type"] == "gender":
            lvl = levelNameToInt(row["HS Level"])
            colLvlGenderHate[-1] = lvl
        elif row["type"] == "countries":
            lvl = levelNameToInt(row["HS Level"])
            colLvlCountriesHate[-1] = lvl
        elif row["type"] == "political":
            lvl = levelNameToInt(row["HS Level"])
            colLvlPoliticalHate[-1] = lvl
        elif row["type"] == "religion":
            lvl = levelNameToInt(row["HS Level"])
            colLvlReligionHate[-1] = lvl
        else:
            raise AssertionError(f"Unsupported type of annotation '{row["type"]}'.")

#add neutral texts
for df in dfNeutralAll:
    for row in df.iter_rows(named=True):
        colText.append(row["Comment"])

        colLvlEthnicityHate.append(0)
        colLvlGenderHate.append(0)
        colLvlCountriesHate.append(0)
        colLvlPoliticalHate.append(0)
        colLvlReligionHate.append(0)

#build dataframe
dfHatePreLblMerge = pl.DataFrame({
    "text": colText,
    "lvl_ethnicity_hate": colLvlEthnicityHate,
    "lvl_gender_hate": colLvlGenderHate,
    "lvl_countries_hate": colLvlCountriesHate,
    "lvl_political_hate": colLvlPoliticalHate,
    "lvl_religion_hate" : colLvlReligionHate
})

#print some stats
print(f"Shape of table pre-cleanup: {dfHatePreLblMerge.shape}.")

In [ ]:
#merge into binary labels
colText = []
colLvlGeneralHate = []

for row in dfHatePreLblMerge.iter_rows(named=True):
    colText.append(row["text"].strip())

    hateColNames = [x for x in row.keys() if x.endswith("_hate")]
    hateColVals = [v for (k, v) in row.items() if k in hateColNames]
    
    hateLvl = (1 if max(hateColVals) > 0 else 0)
    colLvlGeneralHate.append(hateLvl)

#build new dataframe
dfHate = pl.DataFrame({
    "text": colText,
    "lvl_general_hate": colLvlGeneralHate    
})

#print some stats
print(f"Shape of table post label merge: {dfHate.shape}.")

In [ ]:
#merge down and cleanup duplicate samples

mergedRows = []

dfDiscards = pl.DataFrame(data = None, schema={
	"survivor_idx" : pl.Int32,
	"dupe_idx": pl.Int32,
	"primary_lvl_general_hate" : pl.Int32,
	"dupe_lvl_general_hate" : pl.Int32,
	"text" : str
})

dfHate = dfHate.with_row_index("row_idx")
usedUpIdxs = set()

for row in tqdm(dfHate.iter_rows(named=True), desc="Merging down and cleaning"):
	#row already used up? ignore it
	if not(row["row_idx"] in usedUpIdxs):
		#mark survivor as used up
		usedUpIdxs.add(row["row_idx"])

		#find dupes, if any
		dfDupes = dfHate.filter(
			pl.col("text").str.to_lowercase() == row["text"].lower(),
			~pl.col("row_idx").is_in(usedUpIdxs)
		)

		if len(dfDupes) != 0:
			for dupeRow in dfDupes.iter_rows(named=True):
				#find if dupe is conflicting with current row
				isConflict = (
					(row["lvl_general_hate"] != 0 and dupeRow["lvl_general_hate"] != 0)
				)

				#conflict? log it
				if isConflict:
					dfDiscards.vstack(
						pl.from_dicts([{
							"survivor_idx" : row["row_idx"],
							"dupe_idx": dupeRow["row_idx"],
							"primary_lvl_general_hate" : row["lvl_general_hate"],
							"dupe_lvl_general_hate" : dupeRow["lvl_general_hate"],
							"text" : row["text"]
						}], schema=dfDiscards.schema),                       
						in_place=True
					)                   
				
				#merge dupe into survivor by favoring highest hate level
				row["lvl_general_hate"] = max(row["lvl_general_hate"], dupeRow["lvl_general_hate"])

				#mark dupe row index as used up
				usedUpIdxs.add(dupeRow["row_idx"])

		#save survivor
		mergedRows.append(row)

#rebuild the dataframe with merged rows only
dfHate = pl.from_dicts(mergedRows, schema=dfHate.schema).drop("row_idx")

#print some stats
print(f"Number of conflicts found: {len(dfDiscards)}")
print(f"Shape of table post-cleanup: {dfHate.shape}.")

#save results
dfHate.write_parquet(os.path.join(preprocBasePath, "post-cleanup.parquet"))
dfDiscards.write_parquet(os.path.join(preprocBasePath, "cleanup-conflicts.parquet"))

In [ ]:
#load cached post-cleanup results

import polars as pl
dfHate = pl.read_parquet(os.path.join(preprocBasePath, "post-cleanup.parquet"))

In [ ]:
#build the label set
import numpy as np

#define label list
labelLst : List[str] = []

#exract actual labels from the dataset
colLst = ["lvl_general_hate"]
prefixLst = ["general"]

for prefix, col in zip(prefixLst, colLst):
    #get levels present in column
    lvlsPresent = dfHate.select(pl.col(col).unique()).to_series(0).to_list()
    lvlsPresent.sort()

    #build level indices and remap levels to indices in the dataset
    lvlIdx = np.arange(len(lvlsPresent), dtype=np.int32)
    for lvlIdx, lvl in zip(lvlIdx, lvlsPresent):
        dfHate = dfHate.with_columns(
            pl.when(pl.col(col) == lvl)
            .then(lvlIdx)
            .otherwise(pl.col(col))
            .alias(col)
        )

    #register level labels
    for lvl in lvlsPresent:
        labelLst.append(f"{prefix}_{lvl}")

#build label dictionary
labelDict = { idx: lbl for idx, lbl in enumerate(labelLst) }
display(labelDict)

#extract head label counts
numHeadLabels : List[int] = []

for col in colLst:
    lvlsPresent = dfHate.select(pl.col(col).unique()).to_series(0).to_list()
    numHeadLabels.append(len(lvlsPresent))

display(numHeadLabels)

In [ ]:
#build chunked corpus

from corpus.chunkedCorpus import ChunkedCorpus, ChunkedText, Chunk, ChunkSplitter
from typing import List

#label to assign to neutral chunks
labelNone = -1

#
corpus = ChunkedCorpus(labelDefs = labelDict, labelGrps=numHeadLabels, texts = list())
for rowIdx, row in tqdm(enumerate(dfHate.iter_rows(named=True)), "Input texts processed"):
	text = row["text"]

	#skip empty texts, just in case
	if text.strip() == 0:
		continue

	#build chunks from the whole of text
	chunks : List[Chunk] = []
	
	#general hate
	label = f"general_{row["lvl_general_hate"]}"
	labelIdx = [idx for idx, lbl in labelDict.items() if lbl == label][0]
	chunk = Chunk(start=0, len=len(text), label=labelIdx)
	chunks.append(chunk)

	#build the chunked text and register with the corpus
	chunkedText = ChunkedText(id=rowIdx, text=text, chunks = chunks)
	corpus.texts.append(chunkedText)

In [ ]:
#build mixed corpus

#build loaded and neutral sets of samples
lblsNeutral = np.roll(np.array(corpus.labelGrps, np.int32).cumsum(), 1)
lblsNeutral[0] = 0
lblsNeutral = set(lblsNeutral)

loadedSampleIdxs = set()
neutralSampleIdxs = set()

for sampleIdx, sample in enumerate(corpus.texts):
	sampleLbls = set([x.label for x in sample.chunks])
	if sampleLbls == lblsNeutral:
		neutralSampleIdxs.add(sampleIdx)
	else:
		loadedSampleIdxs.add(sampleIdx)

#define mimum and maximum amount of samples to include in the mix
minSamplesInTheMix = 3
maxSamplesInTheMix = 4

#define desired number of mixed samples
maxNumMixedSamples = len(corpus.texts)

#
mixSampCorpus = ChunkedCorpus(labelDefs=corpus.labelDefs, labelGrps=corpus.labelGrps)

#build mixed samples
unusedLoadedSampleIdxs = list(loadedSampleIdxs)
unusedNeutralSampleIdxs = list(neutralSampleIdxs)

from random import Random
rng = Random(0)

mixSampIdx = 0
prg = tqdm(desc="Building mixed samples", total=maxNumMixedSamples)

while mixSampIdx < maxNumMixedSamples:
	#generate amount of samples to include in the mix
	numSamplesInTheMix = rng.choice(range(minSamplesInTheMix, maxSamplesInTheMix+1))

	#snapshot current sample consumption state
	savedUnusedLoadedSampleIdxs = list(unusedLoadedSampleIdxs)
	savedUnusedNeutralSampleIdxs = list(unusedNeutralSampleIdxs)

	#choose chunks for the next mixed sample	
	mixSampChunkIdxs = []
	chunkIsLoaded = np.zeros((numSamplesInTheMix,), np.int32)
	for chunkIdx in range(numSamplesInTheMix):
		#decide what kind of sample to use next
		useLoadedSample = rng.choice([True, False])

		#choose next sample, add to result
		if useLoadedSample:
			sampleIdx = rng.choice(unusedLoadedSampleIdxs)
			unusedLoadedSampleIdxs.remove(sampleIdx)
			mixSampChunkIdxs.append(sampleIdx)

			#indicate we have used a loaded chunk
			chunkIsLoaded[chunkIdx] = 1 
		else:
			sampleIdx = rng.choice(unusedNeutralSampleIdxs)
			unusedNeutralSampleIdxs.remove(sampleIdx)
			mixSampChunkIdxs.append(sampleIdx)

		#restore sample source sets, if any became exhausted
		if len(unusedLoadedSampleIdxs) == 0:
			unusedLoadedSampleIdxs = list(loadedSampleIdxs)
		if len(unusedNeutralSampleIdxs) == 0:
			unusedNeutralSampleIdxs = list(neutralSampleIdxs)

	#did we produce a uniform sample? reject it
	isUniformSample = (chunkIsLoaded.sum() == 0) or (chunkIsLoaded.sum() == numSamplesInTheMix)
	
	if isUniformSample:
		#restore consumption state of base samples
		unusedLoadedSampleIdxs = savedUnusedLoadedSampleIdxs
		unusedNeutralSampleIdxs = savedUnusedNeutralSampleIdxs

	#sample is non-uniform, use it
	else:
		#build unified sample from the chunks
		offset = 0
		mixSampText = ""
		mixSampChunks = []

		for msChunkIdx in mixSampChunkIdxs:
			chunkSample = corpus.texts[msChunkIdx]

			#add space between last sample and new sample, if this is not first sample
			if mixSampText != "":
				mixSampText += " "
				offset +=1 

			#append text of current sample
			mixSampText += chunkSample.text

			#offset and take chunks from the current sample
			for chunk in chunkSample.chunks:
				offsetChunk = Chunk(start=chunk.start + offset, len=chunk.len, label=chunk.label)
				mixSampChunks.append(offsetChunk)

			#advance chunk offset for next round
			offset += len(chunkSample.text)

		#create new mixed sample, add to corpus
		mixSampChunkedText = ChunkedText(id=mixSampIdx, text=mixSampText, chunks=mixSampChunks)
		mixSampCorpus.texts.append(mixSampChunkedText)

		#advance to building next sample
		mixSampIdx += 1
		prg.update(1)

#save result
mixSampCorpus.saveToJson(os.path.join(preprocBasePath, "corpus.ranged.json"))